# Workshop: Lakeflow Jobs — Triggers, Dependencies & Orchestration

> *"The RetailHub pipeline works — now automate it. Configure triggers, define task dependencies, handle failures with repair runs, and monitor via system tables."*

**Learning objective:** Build and evolve a real multi-task Lakeflow Job — add `for_each` and conditional tasks, wire a table-update trigger, repair a failed run, and compute a success rate from run history — verifying every change programmatically with the Databricks SDK.

**Expected duration:** ~55 minutes

### Lab Structure

| Section | Focus | Format |
|---------|-------|--------|
| **Section 1: Workshop** | Hands-on job creation in Databricks UI | Guided walkthrough with screenshots |
| **Section 2: Engineering** | Evolve the job: `for_each`, if/else, triggers, repair, monitoring | UI/JSON steps + SDK verification asserts |
| **Closing check** | Two quick exam-style questions | Fill-in with asserts |

> **SOLUTION NOTEBOOK** — full answers for `lab_08_orchestration`.
>
> **Job dependency:** all Section 2 cells require the Section 1 jobs
> (`<your_name>_customer_pipeline`, `<your_name>_orders_pipeline`) to exist, with the Task A–D
> UI changes applied (for_each + condition tasks added, trigger moved to bronze, one run repaired).

## Setup

In [ ]:
%run ../setup/00_setup

## Section 1: Workshop — Creating & Running Jobs in Databricks UI

In this section you will create two Lakeflow Jobs through the Databricks UI:

| Job | Tasks | Purpose |
|-----|-------|---------|
| **customer_pipeline** | `bronze_customer` → `silver_customer` | Ingest and cleanse customer data |
| **orders_pipeline** | `bronze_orders` → `silver_orders` → `gold_daily_orders` → `gold_summary` | Full orders medallion pipeline, triggered by customer updates |

> **Goal:** Create both jobs, configure task dependencies, set up a **Table Update trigger** so the orders pipeline runs automatically when `silver_customers` is updated, then execute the pipeline end-to-end.

### Step 1: Create a New Job — `customer_pipeline`

Navigate to **Jobs & Pipelines** in the left sidebar, then click **Create Job**.

![Create Job](../../assets/images/training_2026/day3/6f524625f8464c29a2572b0a324333a5.webp)

### Step 2: Set the Job Name

Enter a **unique name** for your job (e.g., `<your_name>_customer_pipeline`).

![Job Name](../../assets/images/training_2026/day3/ae8ab8cafdcf4690b5d9665cbf058798.webp)

> **Note:** The job name does not need to be globally unique — each job is identified by a unique **Job ID** assigned automatically by Databricks.

![Job Details — ID & Creator](../../assets/images/training_2026/day3/460925af2b214f93a2252aed0ea86936.webp)

### Step 3: Configure Job Parameters

Add the following **job parameters** so that all tasks share the same catalog and source path:

| Parameter | Value |
|-----------|-------|
| `catalog` | `retailhub_<your_name>` |
| `source_path` | `/Volumes/retailhub_<your_name>/default/datasets` |

![Job Parameters](../../assets/images/training_2026/day3/e24f8fbe50f642e0a28054972140dcb1.webp)

### Step 4: Add the First Task — `bronze_customer`

Click **Add task** and configure:

| Field | Value |
|-------|-------|
| **Task name** | `bronze_customer` |
| **Type** | Notebook |
| **Source** | Workspace |
| **Path** | Path to your `bronze_customers` notebook |
| **Compute** | **Serverless** (default for notebook tasks — no cluster to pick; the training workspace is serverless-first). On classic compute you would choose a **Job cluster**, never a shared all-purpose cluster |

![Add Task](../../assets/images/training_2026/day3/8d2245e891b849e5966aae38a0e33c5f.webp)

![Task Configuration](../../assets/images/training_2026/day3/c89c69190d40463faae2b5186c78c271.webp)

Click **Create task** when done.

### Step 5: Add the Second Task — `silver_customer`

Repeat the same process for `silver_customer`:
- Set **Depends on** → `bronze_customer` (so it runs only after bronze succeeds)
- Point the notebook path to your `silver_customers` notebook

Your completed job should look like this:

![Customer Pipeline — 2 Tasks](../../assets/images/training_2026/day3/8bdeb6d46c284e3c85bf1c04f4b27657.webp)

### Step 6: Create the Second Job — `orders_pipeline`

Create a **new job** following the same steps as above. Name it `<your_name>_orders_pipeline` and add all four medallion tasks with proper dependencies.

Your completed pipeline should look like this:

![Orders Pipeline — 4 Tasks](../../assets/images/training_2026/day3/119b8804bea74938aff23d816140bf4b.webp)

### Step 7: Configure a Table Update Trigger

Navigate to the **Triggers** tab of the `orders_pipeline` job and add a **Table Update** trigger on `silver_customers`.

This means the orders pipeline will **start automatically** whenever the `silver_customers` table is updated by the customer pipeline.

![Table Update Trigger Configuration](../../assets/images/training_2026/day3/d4dd6298fc9b41e9bc784878ccd8dc3a.webp)

### Step 8: Run the Pipeline

Click **Run now** on the `customer_pipeline` job first.

![Run Job](../../assets/images/training_2026/day3/73685979cbb243b3af4e01fdcadb1a32.webp)

> **Expected result:** The customer pipeline completes successfully, which updates `silver_customers`. This triggers the orders pipeline automatically — both jobs should show a successful run in the **Run History** tab.

### Reference: YAML Definition — `orders_pipeline`

> The YAML below shows the **Declarative Automation Bundle** (formerly Databricks Asset Bundles) definition for the orders pipeline. This is the programmatic equivalent of what you configured in the UI above.

```yaml
resources:
  jobs:
    demo_orders_pipeline:
      name: demo_orders_pipeline
      trigger:
        pause_status: UNPAUSED
        table_update:
          table_names:
            - retailhub_trainer.silver.silver_customers
      tasks:
        - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/.../bronze_orders
            source: WORKSPACE
        - task_key: silver_orders
          depends_on:
            - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/.../silver_orders
            source: WORKSPACE
        - task_key: gold_daily_orders
          depends_on:
            - task_key: silver_orders
          notebook_task:
            notebook_path: /Workspace/.../gold_daily_orders
            source: WORKSPACE
        - task_key: gold_customer_orders_summary
          depends_on:
            - task_key: gold_daily_orders
          notebook_task:
            notebook_path: /Workspace/.../gold_customer_orders_summary
            source: WORKSPACE
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_trainer
        - name: source_path
          default: /Volumes/retailhub_trainer/default/datasets
```

### Reference: YAML Definition — `customer_pipeline`

```yaml
resources:
  jobs:
    demo_customer_pipeline:
      name: demo_customer_pipeline
      tasks:
        - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/.../bronze_customers
            source: WORKSPACE
        - task_key: silver_customer
          depends_on:
            - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/.../silver_customers
            source: WORKSPACE
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_trainer
        - name: source_path
          default: /Volumes/retailhub_trainer/default/datasets
```

> **Key differences:** The customer pipeline has **no trigger** (runs manually or on-demand), while the orders pipeline uses a **table_update trigger** on `silver_customers`.

## Section 2: Engineering — Evolve the Job, Verify with the SDK

Every task below follows the same rhythm: **do it in the UI (or JSON), then prove it with code.** The verification cells use the [Databricks SDK for Python](https://docs.databricks.com/dev-tools/sdk-python.html) (`databricks.sdk`), which authenticates automatically inside a notebook.

> **Requires Section 1** — the `<your_name>_customer_pipeline` and `<your_name>_orders_pipeline` jobs must exist.

| Task | Topic | Verified by |
|------|-------|-------------|
| A | `for_each` task over a list parameter | SDK: task type in job config |
| B | Conditional (If/else) task | SDK: `condition_task` present |
| C | Table-update trigger on a Bronze table | SDK: `trigger.table_update` settings |
| D | Break & repair — Repair run | SDK: `repair_history` of the run |
| E | Run history & success rate | `system.lakeflow` (fallback: Jobs API) |

In [ ]:
# -- Provided: SDK client + job lookup helpers (no TODOs here) --
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()  # auto-auth from the notebook context

user_name    = CATALOG.replace(f"{CATALOG_PREFIX}_", "")
CUSTOMER_JOB = f"{user_name}_customer_pipeline"
ORDERS_JOB   = f"{user_name}_orders_pipeline"

def get_job(name):
    """Return the full job object for an exact job name (Section 1 naming)."""
    matches = list(w.jobs.list(name=name))
    if not matches:
        raise ValueError(
            f"No job named '{name}' found. Finish Section 1 first, "
            f"or adjust CUSTOMER_JOB / ORDERS_JOB to your actual job names."
        )
    return w.jobs.get(matches[0].job_id)

print(f"SDK connected as : {w.current_user.me().user_name}")
print(f"Expected jobs    : {CUSTOMER_JOB} | {ORDERS_JOB}")

### Task A: Add a `for_each` Task Over a List Parameter

`for_each` runs a nested task once **per element of a list** — the Jobs-native way to loop (an explicit May-2026 exam objective).

**UI steps — on `<your_name>_orders_pipeline`:**
1. **Job parameters** → add parameter `tables_to_check` with value `["retailhub_<your_name>.bronze.bronze_customers", "retailhub_<your_name>.silver.silver_customers", "retailhub_<your_name>.gold.gold_daily_orders"]` *(fully qualified — the validate notebook calls `spark.table(...)`)*
2. **Add task** → **For each**; Task name: `check_each_table`; **Depends on:** `gold_summary` (or your last task)
3. **Inputs:** `{{job.parameters.tables_to_check}}` — the list to iterate; **Concurrency:** 1
4. Click **Add a task to loop over** → Type **Notebook**; Path: `materials/orchestration/task_01_validate.py` (your workspace copy); Task parameters: `source_table` = `{{input}}` *(each iteration receives one list element as `{{input}}`)* and `min_rows` = `1` *(the notebook's default of 100 would fail on `gold_daily_orders`, which has ~32 rows — one per day)*

**JSON equivalent** (Job → ⋮ → *Edit as JSON* — this is what the UI generates):
```json
{
  "task_key": "check_each_table",
  "depends_on": [{"task_key": "gold_summary"}],
  "for_each_task": {
    "inputs": "{{job.parameters.tables_to_check}}",
    "concurrency": 1,
    "task": {
      "task_key": "check_one_table",
      "notebook_task": {
        "notebook_path": "/Workspace/.../task_01_validate",
        "base_parameters": {"source_table": "{{input}}", "min_rows": "1"}
      }
    }
  }
}
```

**Guidance — Task A verification**

The SDK mirrors the JSON: each element of `job.settings.tasks` is a `Task` object whose type-specific field (`notebook_task`, `pipeline_task`, `for_each_task`, `condition_task`, ...) is `None` unless configured.

```python
job = get_job(ORDERS_JOB)
loops = [t for t in job.settings.tasks if t.for_each_task is not None]
loops[0].for_each_task.inputs          # the list expression / literal
loops[0].for_each_task.task.task_key   # the nested task
```

In [ ]:
orders_job = get_job(ORDERS_JOB)
foreach_tasks = [t for t in orders_job.settings.tasks if t.for_each_task is not None]

for t in foreach_tasks:
    print(f"for_each task '{t.task_key}' iterating over: {t.for_each_task.inputs}")

In [ ]:
# -- Validation --
all_keys = [t.task_key for t in orders_job.settings.tasks]
assert len(foreach_tasks) >= 1, \
    f"No for_each task found in '{ORDERS_JOB}'. Tasks present: {all_keys}"
fe = foreach_tasks[0].for_each_task
assert fe.inputs, "for_each task must define 'inputs' (the list to iterate)"
assert fe.task is not None, "for_each task must contain a nested task to run per element"
print(f"Task A OK: '{foreach_tasks[0].task_key}' loops a nested "
      f"'{fe.task.task_key}' task over {fe.inputs}")

### Task B: Add a Conditional (If/else) Task

An **If/else condition** task branches the DAG: downstream tasks declare whether they run on the `true` or `false` outcome (the other explicit branching objective).

**UI steps — on `<your_name>_orders_pipeline`:**
1. **Add task** → **If/else condition**; Task name: `check_row_count`; **Depends on:** your `for_each` task (or `gold_summary`)
2. **Condition:** left value `{{job.parameters.min_rows_ok}}`, operator `==`, right value `true`
   *(add job parameter `min_rows_ok` = `true` — in production the left side would be a task value like `{{tasks.gold_summary.values.row_count}}`)*
3. Add a downstream Notebook task (e.g., `send_report`) and set **Depends on:** `check_row_count` **(true)** — it runs only on the true branch

**JSON equivalent:**
```json
{
  "task_key": "check_row_count",
  "depends_on": [{"task_key": "check_each_table"}],
  "condition_task": {
    "op": "EQUAL_TO",
    "left": "{{job.parameters.min_rows_ok}}",
    "right": "true"
  }
}
```

In [ ]:
orders_job = get_job(ORDERS_JOB)
condition_tasks = [t for t in orders_job.settings.tasks if t.condition_task is not None]

for t in condition_tasks:
    ct = t.condition_task
    print(f"condition task '{t.task_key}': {ct.left} {ct.op.value if ct.op else '?'} {ct.right}")

In [ ]:
# -- Validation --
assert len(condition_tasks) >= 1, \
    f"No If/else condition task found in '{ORDERS_JOB}'. " \
    f"Tasks: {[t.task_key for t in orders_job.settings.tasks]}"
ct = condition_tasks[0].condition_task
assert ct.left and ct.right and ct.op, "Condition needs left value, operator, and right value"
dependents = [
    t.task_key for t in orders_job.settings.tasks
    if t.depends_on and any(d.task_key == condition_tasks[0].task_key for d in t.depends_on)
]
print(f"Task B OK: '{condition_tasks[0].task_key}' branches the DAG "
      f"(downstream: {dependents or 'none yet — add a true-branch task!'})")

### Task C: Table-Update Trigger on a Bronze Table

Section 1 put a table-update trigger on `silver_customers`. Move the reaction **one layer earlier**: trigger the orders pipeline as soon as **`bronze_customers`** is updated.

**UI steps — on `<your_name>_orders_pipeline`:**
1. Job page → **Schedules & Triggers** → edit the trigger
2. Type: **Table update**; Table: `retailhub_<your_name>.bronze.bronze_customers`
3. Save. (Optional: set *Minimum time between triggers* to avoid trigger storms.)

> **Exam angle:** know all trigger families — Scheduled (cron), **File arrival**, **Table update**, Continuous, Manual — and when each fits.

In [ ]:
orders_job = get_job(ORDERS_JOB)
trigger = orders_job.settings.trigger
# The SDK field is `table_update` in newer databricks-sdk versions and `table` in older ones
# (the serverless environment may ship an older SDK) -> accept both.
table_trigger = (getattr(trigger, "table_update", None) or getattr(trigger, "table", None)) if trigger else None
table_names = table_trigger.table_names if table_trigger else []

print(f"Trigger watches: {table_names}")

In [ ]:
# -- Validation --
# SDK field name differs by databricks-sdk version: `table_update` (new) / `table` (old)
table_trigger = (getattr(trigger, "table_update", None) or getattr(trigger, "table", None)) if trigger else None
assert trigger is not None, f"'{ORDERS_JOB}' has no trigger configured"
assert table_trigger is not None, "Expected a TABLE UPDATE trigger (not cron/file-arrival)"
assert len(table_names) >= 1, "Table-update trigger must watch at least one table"
assert any(t.lower().startswith(CATALOG.lower()) for t in table_names), \
    f"Trigger should watch a table in YOUR catalog {CATALOG}: {table_names}"
assert any(".bronze." in t.lower() and "bronze_customers" in t.lower() for t in table_names), \
    f"Point the trigger at {CATALOG}.bronze.bronze_customers — currently: {table_names}"
print(f"Task C OK: orders pipeline fires on updates to {table_names}")

### Task D: Break & Repair — Fix a Failed Run Without Re-Running Everything

A **Repair run** re-executes only the failed task and everything downstream of it — succeeded tasks are skipped. Time to see it live.

**Steps — on `<your_name>_orders_pipeline`:**
1. **Break it** *(trainer may have done this for you)*: edit the `gold_summary` task → change its **notebook path** to a name that does not exist, e.g. append `_BROKEN` (`.../gold_customer_orders_summary_BROKEN`)
   > ⚠️ Do **not** try to break it with a task parameter such as `catalog = retailhub_broken`: **job parameters override task parameters with the same key**, so the task would still receive the job-level `catalog` and succeed.
2. **Run now** → watch `bronze_orders` / `silver_orders` / `gold_daily_orders` succeed and `gold_summary` **fail** (downstream tasks: *Upstream failed*)
3. **Fix it**: restore the original notebook path
4. On the failed run's page → **Repair run** → select the failed task → confirm. Only `gold_summary` (and downstream tasks) re-run
5. The run should now finish green

**Guidance — Task D verification**

`w.jobs.list_runs(job_id=...)` lists runs newest-first; `w.jobs.get_run(run_id)` returns the full run including **`repair_history`** — a list with one `ORIGINAL` entry plus one entry per repair. A run that was repaired therefore has `len(repair_history) > 1`.

```python
for r in w.jobs.list_runs(job_id=job_id, limit=25):
    full = w.jobs.get_run(r.run_id)
    if full.repair_history and len(full.repair_history) > 1:
        ...  # this run was repaired
```

In [ ]:
repaired_runs = []
for r in w.jobs.list_runs(job_id=orders_job.job_id, limit=25):
    full = w.jobs.get_run(r.run_id)
    if full.repair_history and len(full.repair_history) > 1:
        repaired_runs.append(full)

for run in repaired_runs:
    print(f"run_id={run.run_id}: {len(run.repair_history)} repair-history entries "
          f"(1 original + {len(run.repair_history) - 1} repair(s)), "
          f"final state: {run.state.result_state}")

In [ ]:
# -- Validation --
assert len(repaired_runs) >= 1, (
    "No repaired run found. Break the gold_summary task, run the job, fix the "
    "parameter, then use 'Repair run' on the failed run — and re-run this cell."
)
rr = repaired_runs[0]
repair_entries = [h for h in rr.repair_history if h.type and h.type.value == "REPAIR"]
assert len(repair_entries) >= 1, "repair_history should contain at least one REPAIR entry"
print(f"Task D OK: run {rr.run_id} was repaired {len(repair_entries)} time(s) — "
      "succeeded tasks were NOT re-executed")

### Task E: Run History & Success Rate — System Tables (with API Fallback)

Compute the job's success rate from **`system.lakeflow.job_run_timeline`** — the SQL-queryable run history used for monitoring dashboards. System tables must be enabled by an account admin and can lag a few minutes, so wrap the query in `try/except` and fall back to the Jobs API.

| Source | Access | Freshness |
|--------|--------|-----------|
| `system.lakeflow.job_run_timeline` | SQL, needs SELECT grant on the schema | Minutes of lag |
| `w.jobs.list_runs()` (Jobs API) | Always available to the job owner | Real-time |

In [ ]:
result_states, source = [], None

try:
    rows = spark.sql(f"""
        SELECT result_state
        FROM system.lakeflow.job_run_timeline
        WHERE job_id = '{orders_job.job_id}'
          AND result_state IS NOT NULL
        ORDER BY period_end_time DESC
        LIMIT 50
    """).collect()
    result_states = [r["result_state"] for r in rows]
    source = "system.lakeflow.job_run_timeline"
    if not result_states:
        raise ValueError("system table returned no rows yet (ingestion lag)")
except Exception as e:
    print(f"[INFO] System-table path unavailable ({type(e).__name__}: {e})")
    print("[INFO] Falling back to the Jobs API — same data, real-time.")
    result_states = [
        r.state.result_state.value
        for r in w.jobs.list_runs(job_id=orders_job.job_id, limit=50)
        if r.state and r.state.result_state
    ]
    source = "Jobs API (w.jobs.list_runs)"

total     = len(result_states)
succeeded = sum(1 for s in result_states if s == "SUCCESS" or s == "SUCCEEDED")
success_rate = succeeded / total * 100 if total else 0.0
print(f"Source: {source}")
print(f"Runs analyzed: {total} | Succeeded: {succeeded} | Success rate: {success_rate:.1f}%")

In [ ]:
# -- Validation --
assert total >= 2, \
    f"Expected at least 2 finished runs (Section 1 run + Task D break/repair), got {total}"
assert succeeded >= 1, "Expected at least one successful run"
assert 0 <= success_rate <= 100
assert source is not None
print(f"Task E OK: success rate {success_rate:.1f}% over {total} runs (source: {source}) — "
      "the Task D failure should keep this below 100%")

## Closing Check — Two Exam Classics

Two quick fill-ins retained from the concept quiz: cron syntax and compute selection.

> **Note:** Lakeflow Jobs schedules use **Quartz cron** syntax — 6 or 7 fields: `seconds minutes hours day-of-month month day-of-week [year]`, e.g. `0 0 6 * * ?` = daily at 06:00. Exactly one of day-of-month / day-of-week must be `?`. The classic 5-field Unix cron (`0 6 * * *`) is **not** accepted by the Jobs scheduler.

In [ ]:
cron_weekdays_8am = "0 0 8 ? * MON-FRI"   # Quartz: sec min hour dom month dow

nightly_etl     = "job_cluster"
interactive_dev = "all_purpose"


In [ ]:
# -- Validation --
_f = cron_weekdays_8am.split()
assert len(_f) in (6, 7), f"Quartz cron has 6-7 fields (seconds first), got {len(_f)}: '{cron_weekdays_8am}'"
assert _f[0] == "0" and _f[1] == "0" and _f[2] == "8", "seconds=0, minutes=0, hours=8"
assert _f[3] == "?" and _f[4] == "*", "day-of-month must be '?' when day-of-week is set; month '*'"
assert _f[5].upper() in ("MON-FRI", "2-6"), "day-of-week: MON-FRI (Quartz numbering 2-6, 1 = SUN)"
assert nightly_etl == "job_cluster", "Scheduled ETL -> job cluster (cheaper, auto-terminates)"
assert interactive_dev == "all_purpose", "Interactive work -> all-purpose (stays running, shared)"
print("Closing check OK: Quartz cron + compute selection correct")


## Summary

### Section 1 — Workshop
You created two Lakeflow Jobs through the Databricks UI, configured task dependencies, set up a **Table Update trigger**, and executed the full pipeline end-to-end.

### Section 2 — Engineering
You evolved the job like a production engineer — every change verified programmatically:

| Task | Skill | Key Takeaway |
|------|-------|-------------|
| A | `for_each` task | Loop a nested task over a list (`{{input}}` per element) — Jobs-native iteration |
| B | If/else condition | `condition_task` branches the DAG; dependents choose the true/false outcome |
| C | Table-update trigger | Jobs can react to Delta table changes — no polling, no cron |
| D | Repair run | Re-runs only the failed task + downstream; `repair_history` records it |
| E | Run monitoring | `system.lakeflow.job_run_timeline` for SQL monitoring; Jobs API as real-time fallback |

> **Exam Tip:** Know **when** to use each trigger type, that `for_each`/conditional tasks are first-class task types, how repair runs scope re-execution (failed + downstream only), and that job clusters — not all-purpose clusters — are the answer for scheduled workloads.

← [08 — Job Orchestration](../day3/demo/08_job_orchestration.ipynb) | **[ README](../../README.md)** | [09 — CI/CD & Automation →](../day3/demo/09_cicd_and_automation.ipynb)